# Notebook 02 — Data Cleaning (ETL Pipeline)

**Project:** VC Investments Analytics  
**Phase:** Week 1 — ETL & Cleaning  
**Input:** `data/raw/investments_VC.csv`  
**Output:** `data/processed/investments_VC_cleaned.csv`

### Cleaning Steps Summary
1. Load raw file with correct encoding
2. Strip whitespace from column names
3. Drop fully-null rows (CSV export artefact)
4. Remove duplicate company records
5. Clean & cast `funding_total_usd` to numeric
6. Parse date columns to `datetime`
7. Strip formatting from `category_list` and `market`
8. Fill funding round columns — NaN → 0
9. Cast `funding_rounds` to integer
10. Validate and save cleaned dataset

## 0. Imports & Paths

In [3]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW_PATH       = '../data/raw/investments_VC.csv'
PROCESSED_PATH = '../data/processed/investments_VC_cleaned.csv'

# Ensure output directory exists
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)

print('Paths configured.')
print('  Input :', os.path.abspath(RAW_PATH))
print('  Output:', os.path.abspath(PROCESSED_PATH))

Paths configured.
  Input : /Users/krishnaverma/Desktop/A_G11_StartupInvestments/data/raw/investments_VC.csv
  Output: /Users/krishnaverma/Desktop/A_G11_StartupInvestments/data/processed/investments_VC_cleaned.csv


---
## Step 1 — Load Raw Data

**Why:** The file contains Latin-1 encoded characters (e.g. `°` in city names).  
Using the default UTF-8 decoder raises a `UnicodeDecodeError` at row ~400.

In [4]:
df = pd.read_csv(RAW_PATH, encoding='latin-1')

print(f'Loaded  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
df.head(3)

Loaded  →  54,294 rows  ×  39 columns


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Step 2 — Strip Whitespace from Column Names

**Why:** Two column names contain leading/trailing spaces (`' market '`, `' funding_total_usd '`),  
which causes silent key-errors and breaks downstream joins.

In [5]:
# Log columns with padding before fixing
padded = [c for c in df.columns if c != c.strip()]
print('Columns with surrounding whitespace (before):', padded)

df.columns = df.columns.str.strip()

print('Column names after strip:')
print(df.columns.tolist())

Columns with surrounding whitespace (before): [' market ', ' funding_total_usd ']
Column names after strip:
['permalink', 'name', 'homepage_url', 'category_list', 'market', 'funding_total_usd', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']


---
## Step 3 — Drop Fully-Null Rows

**Why:** 4,856 rows contain `NaN` in every single column. These are blank rows  
introduced when the CSV was exported and carry no analytical value.

In [6]:
rows_before = len(df)
df = df.dropna(how='all').reset_index(drop=True)
rows_after = len(df)

print(f'Rows before : {rows_before:,}')
print(f'Rows dropped: {rows_before - rows_after:,}  (fully-null rows)')
print(f'Rows after  : {rows_after:,}')

Rows before : 54,294
Rows dropped: 4,856  (fully-null rows)
Rows after  : 49,438


---
## Step 4 — Remove Duplicate Records

**Why:** After removing null rows, 2 duplicate `permalink` entries remain.  
`permalink` is the unique identifier for each company; duplicates would distort  
company-level aggregations and KPI counts.

In [7]:
dups = df[df['permalink'].duplicated(keep=False)]
if not dups.empty:
    print('Duplicate permalink rows:')
    print(dups[['permalink', 'name', 'status', 'funding_total_usd']].to_string())
else:
    print('No duplicate permalinks found.')

Duplicate permalink rows:
                                            permalink                              name     status funding_total_usd
33939                             /organization/prysm                             Prysm  operating     29,30,80,123 
33940                             /organization/prysm                             Prysm  operating     29,30,80,123 
44033  /organization/treasure-valley-urology-services  Treasure Valley Urology Services  operating         3,32,194 
44034  /organization/treasure-valley-urology-services  Treasure Valley Urology Services  operating         3,32,194 


In [8]:
rows_before = len(df)
df = df.drop_duplicates(subset='permalink', keep='first').reset_index(drop=True)
print(f'Rows dropped (duplicate permalink): {rows_before - len(df)}')
print(f'Rows remaining: {len(df):,}')

Rows dropped (duplicate permalink): 2
Rows remaining: 49,436


---
## Step 5 — Clean & Cast `funding_total_usd`

**Why:** The column is stored as a locale-formatted string:  
- Numbers use commas as thousand separators (e.g. `' 17,50,000 '`)  
- Missing / zero-funding entries are represented as `'-'`  

It must be cast to `float` for any financial KPI computation.

In [9]:
# 5a. Strip surrounding whitespace from values
df['funding_total_usd'] = df['funding_total_usd'].str.strip()

# 5b. Replace the dash placeholder with NaN
#     A dash means "no funding data recorded" — not confirmed zero.
dash_count = (df['funding_total_usd'] == '-').sum()
df['funding_total_usd'] = df['funding_total_usd'].replace('-', np.nan)
print(f'Dash placeholders replaced with NaN: {dash_count:,}')

# 5c. Remove all commas and cast to float
df['funding_total_usd'] = (
    df['funding_total_usd']
    .str.replace(',', '', regex=False)
    .astype(float)
)

print(f'funding_total_usd dtype : {df["funding_total_usd"].dtype}')
print(f'Null count after cast   : {df["funding_total_usd"].isnull().sum():,}')
print(f'Non-null stats:')
print(df['funding_total_usd'].describe().round(0).to_string())

Dash placeholders replaced with NaN: 8,531
funding_total_usd dtype : float64
Null count after cast   : 8,531
Non-null stats:
count    4.090500e+04
mean     1.590613e+07
std      1.686773e+08
min      1.000000e+00
25%      3.500000e+05
50%      2.000000e+06
75%      1.000000e+07
max      3.007950e+10


---
## Step 6 — Parse Date Columns

**Why:** `founded_at`, `first_funding_at`, and `last_funding_at` are plain strings.  
Parsing them to `datetime` enables time-series analysis, funding tenure calculations,  
and proper ordering in Tableau.

In [10]:
date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']

for col in date_cols:
    null_before = df[col].isnull().sum()
    df[col] = pd.to_datetime(df[col], errors='coerce')
    null_after = df[col].isnull().sum()
    coerced = null_after - null_before
    print(f'{col:20s}  dtype={df[col].dtype}  '
          f'nulls={null_after:,}  '
          f'(+{coerced} coerced unparseable strings to NaT)')

founded_at            dtype=datetime64[us]  nulls=10,884  (+0 coerced unparseable strings to NaT)
first_funding_at      dtype=datetime64[us]  nulls=0  (+0 coerced unparseable strings to NaT)
last_funding_at       dtype=datetime64[us]  nulls=0  (+0 coerced unparseable strings to NaT)


---
## Step 7 — Clean `category_list` and `market`

**Why:**  
- `category_list` values are wrapped in leading and trailing pipe characters (e.g. `|Tech|Finance|`) — the outer pipes are formatting artefacts with no semantic meaning.  
- `market` values have surrounding whitespace that would create false duplicates in group-by operations.

In [11]:
# 7a. Strip outer pipe characters from category_list
df['category_list'] = df['category_list'].str.strip('|').str.strip()

print('category_list — sample after cleaning:')
print(df['category_list'].dropna().head(5).tolist())

category_list — sample after cleaning:
['Entertainment|Politics|Social Media|News', 'Games', 'Publishing|Education', 'Electronics|Guides|Coffee|Restaurants|Music|iPhone|Apps|Mobile|iOS|E-Commerce', 'Tourism|Entertainment|Games']


In [12]:
# 7b. Strip surrounding whitespace from market values
before_unique = df['market'].nunique()
df['market'] = df['market'].str.strip()
after_unique = df['market'].nunique()

print(f'market unique values  →  before: {before_unique}  |  after: {after_unique}')
print(f'Duplicates collapsed by stripping: {before_unique - after_unique}')

market unique values  →  before: 753  |  after: 753
Duplicates collapsed by stripping: 0


---
## Step 8 — Fill Funding Round Columns (NaN → 0)

**Why:** The funding round columns (`seed`, `venture`, `angel`, `round_A` … `round_H`, etc.)  
contain `NaN` where a company did not participate in that round.  
These are structural zeros — the absence of a record means no funding of that type,  
not missing data. Filling with 0 allows correct summation and comparison.

In [13]:
funding_round_cols = [
    'seed', 'venture', 'equity_crowdfunding', 'undisclosed',
    'convertible_note', 'debt_financing', 'angel', 'grant',
    'private_equity', 'post_ipo_equity', 'post_ipo_debt',
    'secondary_market', 'product_crowdfunding',
    'round_A', 'round_B', 'round_C', 'round_D',
    'round_E', 'round_F', 'round_G', 'round_H'
]

nulls_before = df[funding_round_cols].isnull().sum().sum()
df[funding_round_cols] = df[funding_round_cols].fillna(0)
nulls_after = df[funding_round_cols].isnull().sum().sum()

print(f'Null values in funding round columns  →  before: {nulls_before:,}  |  after: {nulls_after}')

Null values in funding round columns  →  before: 0  |  after: 0


---
## Step 9 — Cast `funding_rounds` to Integer

**Why:** Number of funding rounds is a count and should be an integer.  
It was loaded as `float64` because of the null-row contamination in Step 3.  
Rows where this remains null (no round data) are left as `NaN` using the  
nullable `Int64` dtype.

In [14]:
df['funding_rounds'] = df['funding_rounds'].astype('Int64')
print(f'funding_rounds dtype : {df["funding_rounds"].dtype}')
print(df['funding_rounds'].value_counts().sort_index().head(10))

funding_rounds dtype : Int64
funding_rounds
1     32038
2      9219
3      4025
4      1997
5      1001
6       560
7       252
8       152
9        84
10       43
Name: count, dtype: Int64


---
## Step 10 — Post-Cleaning Validation

In [15]:
# ── Shape ──────────────────────────────────────────────────────────────
print('=== Final Dataset Shape ===')
print(f'  Rows : {df.shape[0]:,}')
print(f'  Cols : {df.shape[1]}')

=== Final Dataset Shape ===
  Rows : 49,436
  Cols : 39


In [16]:
# ── Data types ─────────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes.to_string())

=== Data Types ===
permalink                          str
name                               str
homepage_url                       str
category_list                      str
market                             str
funding_total_usd              float64
status                             str
country_code                       str
state_code                         str
region                             str
city                               str
funding_rounds                   Int64
founded_at              datetime64[us]
founded_month                      str
founded_quarter                    str
founded_year                   float64
first_funding_at        datetime64[us]
last_funding_at         datetime64[us]
seed                           float64
venture                        float64
equity_crowdfunding            float64
undisclosed                    float64
convertible_note               float64
debt_financing                 float64
angel                          float64
grant 

In [17]:
# ── Remaining missing values ────────────────────────────────────────────
print('=== Remaining Missing Values (non-zero only) ===')
miss = df.isnull().sum()
miss_pct = (df.isnull().mean() * 100).round(1)
miss_df = pd.concat([miss, miss_pct], axis=1, keys=['count', '%'])
print(miss_df[miss_df['count'] > 0].to_string())
print()
print('Note: remaining nulls in geographic fields (state_code, region, city)')
print('reflect non-US companies where these fields are not applicable.')
print('Nulls in founded_at / founded_year reflect records with no founding date on file.')

=== Remaining Missing Values (non-zero only) ===
                   count     %
name                   1   0.0
homepage_url        3448   7.0
category_list       3960   8.0
market              3967   8.0
funding_total_usd   8531  17.3
status              1314   2.7
country_code        5273  10.7
state_code         19277  39.0
region              5273  10.7
city                6116  12.4
founded_at         10884  22.0
founded_month      10956  22.2
founded_quarter    10956  22.2
founded_year       10956  22.2

Note: remaining nulls in geographic fields (state_code, region, city)
reflect non-US companies where these fields are not applicable.
Nulls in founded_at / founded_year reflect records with no founding date on file.


In [18]:
# ── Duplicate check ─────────────────────────────────────────────────────
dup_remaining = df['permalink'].duplicated().sum()
print(f'Duplicate permalinks remaining: {dup_remaining}')

Duplicate permalinks remaining: 0


In [19]:
# ── funding_total_usd sanity check ──────────────────────────────────────
print('=== funding_total_usd Distribution ===')
print(df['funding_total_usd'].describe().apply(lambda x: f'{x:,.0f}'))
print()
print('Companies with $0 or null funding:', (df['funding_total_usd'].fillna(0) == 0).sum())

=== funding_total_usd Distribution ===
count            40,905
mean         15,906,131
std         168,677,339
min                   1
25%             350,000
50%           2,000,000
75%          10,000,000
max      30,079,503,000
Name: funding_total_usd, dtype: str

Companies with $0 or null funding: 8531


In [20]:
# ── Date range checks ───────────────────────────────────────────────────
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
    print(f'{col:20s}  min={df[col].min().date() if df[col].notna().any() else "N/A"}'
          f'   max={df[col].max().date() if df[col].notna().any() else "N/A"}')

founded_at            min=1636-09-08   max=2014-12-13
first_funding_at      min=0001-05-14   max=2014-12-31
last_funding_at       min=0001-05-14   max=2015-01-01


In [21]:
# ── Final preview ───────────────────────────────────────────────────────
df.head()

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,Entertainment|Politics|Social Media|News,News,1750000.0,acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,Games,Games,4000000.0,operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,Publishing|Education,Publishing,40000.0,operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,Electronics|Guides|Coffee|Restaurants|Music|iP...,Electronics,1500000.0,operating,GBR,NaN,London,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,Tourism|Entertainment|Games,Tourism,60000.0,operating,USA,TX,Dallas,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Step 11 — Save Cleaned Dataset

In [22]:
df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8')

saved_size_mb = os.path.getsize(PROCESSED_PATH) / (1024 ** 2)

print('✅  Cleaned dataset saved successfully.')
print(f'   Path  : {os.path.abspath(PROCESSED_PATH)}')
print(f'   Rows  : {len(df):,}')
print(f'   Cols  : {len(df.columns)}')
print(f'   Size  : {saved_size_mb:.2f} MB')

✅  Cleaned dataset saved successfully.
   Path  : /Users/krishnaverma/Desktop/A_G11_StartupInvestments/data/processed/investments_VC_cleaned.csv
   Rows  : 49,436
   Cols  : 39
   Size  : 13.40 MB


---
## Cleaning Log Summary

| Step | Transformation | Rows / Values Affected |
|------|---------------|------------------------|
| 2 | Stripped whitespace from 2 column names (`market`, `funding_total_usd`) | 2 columns |
| 3 | Dropped fully-null rows | −4,856 rows |
| 4 | Removed duplicate `permalink` entries (kept first occurrence) | −2 rows |
| 5 | Converted `funding_total_usd` from comma-formatted string to `float`; replaced `'-'` with `NaN` | ~8,531 dash values → NaN |
| 6 | Parsed `founded_at`, `first_funding_at`, `last_funding_at` to `datetime64` | 3 columns re-typed |
| 7 | Stripped outer `|` from `category_list`; stripped whitespace from `market` | 2 columns |
| 8 | Filled 21 funding round columns — `NaN` → `0` (structural zeros) | 21 columns |
| 9 | Cast `funding_rounds` from `float64` to nullable `Int64` | 1 column |

**Intentionally not imputed:**  
- `status` — 1,314 nulls: company status is unknown, not inferable  
- `founded_at` / `founded_year` — 15,740 nulls: no reliable basis for imputation  
- `state_code`, `region`, `city` — many nulls belong to non-US companies  
- `funding_total_usd` — dash-derived nulls: recorded as unknown, not zero  

---
*Proceed to `03_eda.ipynb` for exploratory data analysis.*